# SOC Trajectory Visualization: Panasonic25 -> CMU eVTOL Transfer

Per-mission SOC (state of charge) trajectory plots for the seven CMU
"core seven" target cells, comparing the ground-truth reference SOC against
seven transfer-learning curves: Vidal Fine-tuning, DTNN, SFTNN, DARM, and
the three Backbone architectures (CNN-LSTM, TCN, Transformer).

**Each subplot shows exactly one representative test mission for one
battery cell** -- not the whole test-set lifespan (an earlier version
concatenated every test mission chronologically, which was unreadable with
100-450+ missions per cell).

> **Preview status.** These are diagnostic / preview figures, not final
> publication benchmark results. Every currently-available CMU run's own
> `run_metadata.json` sets `formal_benchmark_eligible: false`. DTNN
> trajectory uses formal repeat 1 (see the DTNN section below). Treat every
> curve here as "what the pipeline currently produces", not as a claim
> about final model quality.

**Data availability as of 2026-09-18 (Drive inventory re-verified against
`battery_soc_benchmark/outputs/benchmarks/panasonic_to_cmu_evtol/` -- both
repo runner code and the actual files on Drive, not just default directory
names):**

| Framework | Status | Notes |
|---|---|---|
| SFTNN | available | `sftnn_med_transfer_ns_relabel_v2/test_predictions.parquet`, `method == "med"`, `seed == 42`, `evaluation_unit == all_timesteps` |
| DARM | available | `liu_2025_darm_ns_relabel_v2_e30_clean/test_predictions.parquet`, `method == "DARM"`, `seed == 42`, `evaluation_unit == last_step_per_window` |
| Backbone | available, split into 3 curves | `backbone_transfer_ns_relabel_v2/test_predictions.parquet` holds all three architectures (`model in {cnn_lstm, tcn, transformer_encoder}`) in one file; each is filtered separately (`method == "fine_tuned"`, `budget == "full"`, `seed == 42`) into "Backbone CNN-LSTM" / "Backbone TCN" / "Backbone Transformer" -- an earlier version of this notebook only plotted `transformer_encoder` and called it "Backbone", which under-represented the other two architectures |
| Vidal Fine-tuning | available, legacy schema, loaded through an adapter | `vidal/vidal_full/seed_42/20260916_192416/{cell_id}/test_predictions.parquet` -- a genuine CMU eVTOL full-target fine-tuning run (7/7 cells), previously misjudged as "not run" because its output root and schema don't match the other frameworks' convention (no `method`/`seed`/`timestep_index` columns, per-cell files instead of one aggregated file, `elapsed_candidate_s_end` instead of `elapsed_candidate_s`). See `load_vidal_full_predictions` below. |
| DTNN | available, formal run | `bian_2021/initial_soc_70_v2/formal/test_predictions.parquet` -- previously misjudged as "not run" because the script's own `--output-root` default (`bian_2021_cmu`) was never actually used; the real output root is set inside `33_bian_dtnn_panasonic25_to_cmu.ipynb`. `method == "Bian MDA"`. DTNN trajectory uses formal repeat 1. Its `seed` column actually stores the **repeat index**, not a shared random seed -- it is filtered separately from `PLOT_SEED`. |
| TATN | **omitted** -- no prediction-level output | Six historical run folders on Drive have aggregate `condition_metrics.csv` (proving TATN did run, once, before this repo's prediction/phase-diagnostics were added) but none of them have a `test_predictions.parquet` -- only `results.json` + `target.pt`. There is no per-timestep prediction data to plot. The official `pre-25.pt` checkpoint *is* present on Drive with the expected SHA256, so a fresh TATN run with the current script would produce plottable output, but that has not happened yet. This notebook never fabricates a TATN curve from aggregate metrics, a checkpoint, or interpolation -- TATN is simply absent from the plots until a real prediction-level run exists.

In [ ]:
"""Code Cell 1 -- imports and plotting style."""

import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import gridspec

# Running on Colab: mount Google Drive so DATA_ROOT below resolves.
# No-op (and harmless) outside Colab.
try:
    import google.colab  # noqa: F401

    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linewidth": 0.6,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 10,
    }
)

pd.set_option("display.max_columns", 50)


In [ ]:
"""Code Cell 2 -- centralized configuration.

Every path, column name, filter value, and display choice used below lives
here. This is a re-verified inventory (2026-09-18): Drive was scanned
recursively, repo runner/notebook code was read to find each framework's
REAL output root (several differ from the script's own --output-root
default), and every schema claim below was checked against an actual
run_metadata.json / parquet-column read, not assumed.
"""

# --- Where the benchmark outputs live -------------------------------------
DATA_ROOT = Path(
    "/content/drive/MyDrive/battery_soc_benchmark/outputs/benchmarks/panasonic_to_cmu_evtol"
)

# The CMU "core seven" target cells.
CELL_IDS = ["VAH01", "VAH17", "VAH27", "VAH30", "VAH15", "VAH12", "VAH11"]

# Display order for the legend and for consistent per-framework colors.
# TATN is deliberately NOT in this list -- see the title cell and the DTNN/
# TATN notes below; it has no prediction-level output to plot.
FRAMEWORK_ORDER = [
    "Vidal Fine-tuning",
    "DTNN",
    "SFTNN",
    "DARM",
    "Backbone CNN-LSTM",
    "Backbone TCN",
    "Backbone Transformer",
]

# --- Column names in the unified test_predictions.parquet schema ---------
# (SFTNN / DARM / Backbone / DTNN all use this; Vidal does not -- see
# load_vidal_full_predictions, which adapts Vidal's legacy schema into
# this same set of column names so downstream code never special-cases it.)
COL_CELL_ID = "cell_id"
COL_METHOD = "method"
COL_MODEL = "model"
COL_BUDGET = "budget"
COL_SEED = "seed"
COL_SOC_REFERENCE = "soc_reference"
COL_SOC_PREDICTION = "soc_prediction"
COL_MISSION_ID = "mission_cycle_id"
COL_MISSION_SEQUENCE_ORDER = "mission_sequence_order"
COL_TIMESTEP_INDEX = "timestep_index"
COL_SOURCE_ROW_RIGHT = "source_row_right"
COL_ELAPSED_TIME_CANDIDATES = [
    "elapsed_candidate_s",
    "elapsed_candidate_s_end",
    "original_time_s",
    "timestamp",
]

# --- SFTNN / DARM: generic-loader frameworks -------------------------------
# Both use the unified schema end to end, so they go through the generic
# _read_predictions_file loader with an exact method filter plus PLOT_SEED.
GENERIC_RESULT_FILES: dict[str, Path] = {
    "SFTNN": DATA_ROOT / "sftnn_med_transfer_ns_relabel_v2" / "test_predictions.parquet",
    "DARM": DATA_ROOT / "liu_2025_darm_ns_relabel_v2_e30_clean" / "test_predictions.parquet",
}
FRAMEWORK_FILTER_SPECS: dict[str, dict[str, object]] = {
    "SFTNN": {COL_METHOD: "med"},  # never "zero_shot" / "dann_mmd"
    "DARM": {COL_METHOD: "DARM"},  # never "Source only"
}

# All *generic* frameworks (SFTNN, DARM, Backbone) are shown for this seed.
# DTNN and Vidal are handled separately -- see below.
PLOT_SEED = 42

# --- Backbone: one shared parquet, split into three display curves -------
# backbone_transfer_ns_relabel_v2/test_predictions.parquet holds all three
# architectures in one file. An earlier version of this notebook only kept
# "transformer_encoder" and called the resulting single curve "Backbone",
# silently dropping cnn_lstm/tcn. This version reads the file once and
# splits it into three separate framework entries instead.
BACKBONE_FILE = DATA_ROOT / "backbone_transfer_ns_relabel_v2" / "test_predictions.parquet"
BACKBONE_METHOD = "fine_tuned"  # never "zero_shot" -- see run_backbone_transfer_cmu.py:
# method = "zero_shot" if budget == "zero_shot" else "fine_tuned"
BACKBONE_BUDGET = "full"  # full-target fine-tuning, not a partial/zero_shot/Npct budget
BACKBONE_MODEL_TO_DISPLAY_NAME: dict[str, str] = {
    "cnn_lstm": "Backbone CNN-LSTM",
    "tcn": "Backbone TCN",
    "transformer_encoder": "Backbone Transformer",
}

# --- Vidal: legacy schema, loaded through a dedicated adapter -------------
# Confirmed real run (2026-09-16), 7/7 cells, seed 42 baked into the path
# itself (there is no `seed` column in the file). "vidal_full" is the
# full-target fine-tuning setting -- see run_cmu_transfer.py's SETTINGS
# tuple and run_metadata.json's "setting": "vidal_full".
# To pin a different/newer run, change VIDAL_RUN_ID below (or see
# _latest_vidal_run_id for an auto-discovery fallback, unused by default).
VIDAL_SETTING = "vidal_full"
VIDAL_SEED = 42
VIDAL_RUN_ID = "20260916_192416"
VIDAL_ROOT = DATA_ROOT / "vidal" / VIDAL_SETTING / f"seed_{VIDAL_SEED}" / VIDAL_RUN_ID

# --- DTNN: unified schema, but framework-specific repeat/seed handling ---
# DTNN's real output root (set inside 33_bian_dtnn_panasonic25_to_cmu.ipynb,
# NOT run_bian_cmu.py's own --output-root default of "bian_2021_cmu") is
# bian_2021/initial_soc_70_v2/{screening,formal}. DTNN trajectory uses
# formal repeat 1 (source_epochs=2000, transfer_epochs=2000, 5 repeats
# total). Change DTNN_REPEAT to 1-5 to use a different repeat from the
# formal run instead; nothing else needs to change.
#
# IMPORTANT: DTNN's `seed` column does not hold a shared random seed -- it
# holds the REPEAT index. PLOT_SEED (42) is never applied to DTNN;
# DTNN_REPEAT is used instead.
DTNN_RUN_MODE = "formal"
DTNN_REPEAT = 1
DTNN_FILE = (
    DATA_ROOT / "bian_2021" / "initial_soc_70_v2" / DTNN_RUN_MODE / "test_predictions.parquet"
)
DTNN_METHOD = "Bian MDA"  # never "Source only"

# --- Each framework's own primary-evaluation output convention -----------
# Used only to pick which framework's rows serve as the Reference SOC
# source (see _choose_reference_framework) -- never to interpolate or
# resample any framework's predictions.
EVALUATION_UNIT: dict[str, str] = {
    "Vidal Fine-tuning": "last_step_per_window",
    "DTNN": "all_timesteps",
    "SFTNN": "all_timesteps",
    "DARM": "last_step_per_window",
    "Backbone CNN-LSTM": "all_timesteps",
    "Backbone TCN": "all_timesteps",
    "Backbone Transformer": "all_timesteps",
}

# --- Representative-mission selection --------------------------------------
# "midpoint": deterministic, no manual cherry-picking.
MISSION_SELECTION_MODE = "midpoint"

# Anchor frameworks for common-mission selection (see Code Cell 4 for the
# full rule): with 7 candidate curves of very different sparsity (Vidal/
# DARM are last_step_per_window; DTNN/SFTNN/Backbone are all_timesteps),
# requiring ALL loaded frameworks to share a mission would shrink the
# common set drastically and let the sparsest framework veto everyone
# else's representative mission. Instead, only these high-coverage anchor
# frameworks determine the common mission set; every other framework is
# plotted on that mission if it has data there, and skipped (with a
# warning) otherwise.
MISSION_ANCHOR_FRAMEWORKS = ["SFTNN", "Backbone Transformer"]

# Force a specific mission_sequence_order for a specific cell, bypassing
# the midpoint rule. Empty by default.
MISSION_OVERRIDE: dict[str, int] = {
    # "VAH01": 123,
}

# --- Reference-SOC cross-check -------------------------------------------
REFERENCE_SOC_MISMATCH_TOLERANCE = 1e-3

# --- Display-only downsampling (never alters underlying data) ------------
PLOT_EVERY_N = 1

# --- Output ----------------------------------------------------------------
OUTPUT_DIR = Path("./soc_trajectory_outputs")
FIG_DPI = 300
SAVE_INDIVIDUAL_CELL_FIGURES = False

# --- Colors ------------------------------------------------------------
# One fixed, distinct color per framework across every subplot. The three
# Backbone architectures deliberately use visually distant hues (green /
# cyan / brown) rather than three shades of one color, so they stay
# distinguishable in a dense subplot.
REFERENCE_COLOR = "#1a1a1a"
FRAMEWORK_COLORS: dict[str, str] = {
    "Vidal Fine-tuning": "#1f77b4",
    "DTNN": "#ff7f0e",
    "SFTNN": "#d62728",
    "DARM": "#9467bd",
    "Backbone CNN-LSTM": "#2ca02c",
    "Backbone TCN": "#17becf",
    "Backbone Transformer": "#8c564b",
}


In [ ]:
"""Code Cell 3 -- loading.

Three loading strategies, one per schema:

1. `_read_predictions_file` (generic): SFTNN / DARM. Unified schema, exact
   method filter from FRAMEWORK_FILTER_SPECS, PLOT_SEED applied.
2. `load_backbone_predictions`: reads backbone_transfer_ns_relabel_v2's
   ONE shared parquet a single time, then splits it into three framework
   entries by `model`, all three still `method == "fine_tuned"`,
   `budget == "full"`, `seed == PLOT_SEED`.
3. `load_dtnn_predictions`: unified schema, but seed/repeat handling is
   framework-specific (DTNN_REPEAT, never PLOT_SEED).
4. `load_vidal_full_predictions`: a real adapter for Vidal's legacy,
   per-cell schema (see Code Cell 2's comments) -- maps its columns into
   the same unified names everything else uses, entirely in memory. It
   never touches the original files on Drive.

Every loader warns and skips (never raises) on a missing file, missing
column, or empty filter result, so the notebook stays usable while some
frameworks' data changes.
"""

_REQUIRED_COLUMNS = {
    COL_CELL_ID,
    COL_METHOD,
    COL_SOC_REFERENCE,
    COL_SOC_PREDICTION,
    COL_MISSION_ID,
    COL_MISSION_SEQUENCE_ORDER,
    COL_TIMESTEP_INDEX,
}
_OPTIONAL_COLUMNS = {
    COL_MODEL,
    COL_BUDGET,
    COL_SEED,
    COL_SOURCE_ROW_RIGHT,
    "repeat",
    *COL_ELAPSED_TIME_CANDIDATES,
}


def _parquet_columns(path: Path) -> set[str]:
    """Schema-only read (no row data)."""
    import pyarrow.parquet as pq

    return set(pq.ParquetFile(str(path)).schema.names)


def _ensure_percent_scale(series: pd.Series) -> pd.Series:
    """SOC is stored as a 0-1 fraction in every framework's schema here;
    auto-convert to percent only if the data actually looks fraction-scaled."""
    finite = series.dropna()
    if finite.empty:
        return series * 100.0
    if finite.max() <= 1.5:
        return series * 100.0
    return series


def _read_predictions_file(framework: str, path: Path) -> pd.DataFrame | None:
    """Generic unified-schema loader, used by SFTNN and DARM: exact filter
    from FRAMEWORK_FILTER_SPECS, then PLOT_SEED."""
    if not Path(path).exists():
        warnings.warn(f"[{framework}] file not found at {path} -- skipping.")
        return None
    try:
        available_columns = _parquet_columns(path)
    except Exception as exc:  # noqa: BLE001 -- intentionally broad: never crash the notebook
        warnings.warn(f"[{framework}] could not read schema of {path}: {exc!r} -- skipping.")
        return None

    missing_required = _REQUIRED_COLUMNS - available_columns
    if missing_required:
        warnings.warn(
            f"[{framework}] {path} is missing required columns {missing_required} -- skipping."
        )
        return None

    columns = sorted(_REQUIRED_COLUMNS | (_OPTIONAL_COLUMNS & available_columns))
    try:
        frame = pd.read_parquet(path, columns=columns)
    except Exception as exc:  # noqa: BLE001
        warnings.warn(f"[{framework}] failed reading {path}: {exc!r} -- skipping.")
        return None

    filter_spec = FRAMEWORK_FILTER_SPECS.get(framework, {})
    missing_filter_columns = set(filter_spec) - set(frame.columns)
    if missing_filter_columns:
        warnings.warn(
            f"[{framework}] cannot apply filter {filter_spec} -- missing columns "
            f"{missing_filter_columns} -- skipping."
        )
        return None
    mask = pd.Series(True, index=frame.index)
    for column, value in filter_spec.items():
        mask &= frame[column] == value
    frame = frame[mask]

    if COL_SEED in frame.columns:
        frame = frame[frame[COL_SEED] == PLOT_SEED]

    frame = frame[frame[COL_CELL_ID].isin(CELL_IDS)]
    if frame.empty:
        warnings.warn(f"[{framework}] no rows left after filtering -- skipping.")
        return None

    frame = frame.copy()
    frame[COL_SOC_REFERENCE] = _ensure_percent_scale(frame[COL_SOC_REFERENCE])
    frame[COL_SOC_PREDICTION] = _ensure_percent_scale(frame[COL_SOC_PREDICTION])
    frame["framework"] = framework
    return frame


def load_backbone_predictions(path: Path = BACKBONE_FILE) -> dict[str, pd.DataFrame]:
    """Reads the shared backbone parquet ONCE, then splits it into three
    display frameworks by `model`. All three share the method/budget/seed
    filter; only `model` differs."""
    result: dict[str, pd.DataFrame] = {}
    if not Path(path).exists():
        warnings.warn(f"[Backbone] file not found at {path} -- skipping all three backbone curves.")
        return result
    try:
        available_columns = _parquet_columns(path)
    except Exception as exc:  # noqa: BLE001
        warnings.warn(f"[Backbone] could not read schema of {path}: {exc!r} -- skipping.")
        return result
    missing_required = _REQUIRED_COLUMNS - available_columns
    if missing_required:
        warnings.warn(f"[Backbone] {path} missing required columns {missing_required} -- skipping.")
        return result
    columns = sorted(_REQUIRED_COLUMNS | (_OPTIONAL_COLUMNS & available_columns))
    try:
        raw = pd.read_parquet(path, columns=columns)
    except Exception as exc:  # noqa: BLE001
        warnings.warn(f"[Backbone] failed reading {path}: {exc!r} -- skipping.")
        return result

    if COL_MODEL not in raw.columns or COL_BUDGET not in raw.columns:
        warnings.warn(f"[Backbone] {path} missing model/budget columns -- skipping.")
        return result

    base_mask = (raw[COL_METHOD] == BACKBONE_METHOD) & (raw[COL_BUDGET] == BACKBONE_BUDGET)
    if COL_SEED in raw.columns:
        base_mask &= raw[COL_SEED] == PLOT_SEED
    base = raw[base_mask]

    for model_value, display_name in BACKBONE_MODEL_TO_DISPLAY_NAME.items():
        sub = base[base[COL_MODEL] == model_value]
        sub = sub[sub[COL_CELL_ID].isin(CELL_IDS)]
        if sub.empty:
            warnings.warn(
                f"[{display_name}] no rows for model=={model_value!r}, "
                f"method=={BACKBONE_METHOD!r}, budget=={BACKBONE_BUDGET!r}, "
                f"seed=={PLOT_SEED} -- skipping."
            )
            continue
        sub = sub.copy()
        sub[COL_SOC_REFERENCE] = _ensure_percent_scale(sub[COL_SOC_REFERENCE])
        sub[COL_SOC_PREDICTION] = _ensure_percent_scale(sub[COL_SOC_PREDICTION])
        sub["framework"] = display_name
        result[display_name] = sub
    return result


def load_dtnn_predictions(
    path: Path = DTNN_FILE, repeat: int = DTNN_REPEAT, method: str = DTNN_METHOD
) -> pd.DataFrame | None:
    """DTNN's own loader: unified schema, but the seed-like column holds a
    REPEAT index (this run only has repeat/seed == 1), so PLOT_SEED is
    never applied here -- `repeat` is used instead."""
    if not Path(path).exists():
        warnings.warn(f"[DTNN] file not found at {path} -- skipping.")
        return None
    try:
        available_columns = _parquet_columns(path)
    except Exception as exc:  # noqa: BLE001
        warnings.warn(f"[DTNN] could not read schema of {path}: {exc!r} -- skipping.")
        return None
    missing_required = _REQUIRED_COLUMNS - available_columns
    if missing_required:
        warnings.warn(f"[DTNN] {path} missing required columns {missing_required} -- skipping.")
        return None
    columns = sorted(_REQUIRED_COLUMNS | (_OPTIONAL_COLUMNS & available_columns))
    try:
        frame = pd.read_parquet(path, columns=columns)
    except Exception as exc:  # noqa: BLE001
        warnings.warn(f"[DTNN] failed reading {path}: {exc!r} -- skipping.")
        return None

    frame = frame[frame[COL_METHOD] == method]

    # Prefer an explicit `repeat` column when present; fall back to `seed`
    # (DTNN's schema names it "seed" but the values are repeat indices --
    # see the config-cell comment). Never falls back to PLOT_SEED.
    repeat_like_column = "repeat" if "repeat" in frame.columns else (
        COL_SEED if COL_SEED in frame.columns else None
    )
    if repeat_like_column is not None:
        frame = frame[frame[repeat_like_column] == repeat]
    else:
        warnings.warn(
            "[DTNN] no repeat/seed-like column found -- cannot select a single "
            "repeat; using all rows for the requested method."
        )

    frame = frame[frame[COL_CELL_ID].isin(CELL_IDS)]
    if frame.empty:
        warnings.warn(
            f"[DTNN] no rows left after filtering method=={method!r}, repeat=={repeat} -- skipping."
        )
        return None

    frame = frame.copy()
    frame[COL_SOC_REFERENCE] = _ensure_percent_scale(frame[COL_SOC_REFERENCE])
    frame[COL_SOC_PREDICTION] = _ensure_percent_scale(frame[COL_SOC_PREDICTION])
    frame["framework"] = "DTNN"
    return frame


def load_vidal_full_predictions(root: Path = VIDAL_ROOT) -> pd.DataFrame | None:
    """Adapter for Vidal's legacy per-cell schema (run_cmu_transfer.py).
    Never modifies the original files -- reads each cell's
    test_predictions.parquet and maps it, entirely in memory, onto the same
    unified column names every other loader here produces:

      - framework            <- constant "Vidal Fine-tuning"
      - cell_id               <- cell_id (unchanged)
      - mission_cycle_id       <- mission_cycle_id (unchanged)
      - mission_sequence_order <- mission_sequence_order (unchanged)
      - soc_reference/soc_prediction <- unchanged (already 0-1 fractions)
      - elapsed_candidate_s    <- elapsed_candidate_s_end, RENAMED not
                                   normalized (Vidal records the window's
                                   end-of-window elapsed time; there is no
                                   per-timestep series to preserve, since
                                   Vidal is last_step_per_window)
      - original_time_s        <- original_time_s_end, same rationale
      - source_row_right       <- unchanged, when present (used for the
                                   cross-framework reference consistency
                                   check)
      - timestep_index         <- NOT a real field in this schema (Vidal
                                   evaluates one point per window). A
                                   LOCAL, fallback-only sample index is
                                   synthesized per mission after sorting by
                                   elapsed time, purely so downstream code
                                   that expects a timestep_index column
                                   keeps working -- this is never presented
                                   as a real per-timestep index and is only
                                   ever used as an ordering tiebreaker.
    Vidal has no `method`/`seed` columns to filter by: the setting
    (vidal_full) already means "full-target fine-tuning", and the seed is
    already fixed by VIDAL_ROOT's own seed_{VIDAL_SEED} path segment.
    """
    frames = []
    for cell_id in CELL_IDS:
        path = root / cell_id / "test_predictions.parquet"
        if not path.exists():
            warnings.warn(f"[Vidal Fine-tuning] {path} not found -- skipping {cell_id}.")
            continue
        try:
            raw = pd.read_parquet(path)
        except Exception as exc:  # noqa: BLE001
            warnings.warn(f"[Vidal Fine-tuning] failed reading {path}: {exc!r} -- skipping {cell_id}.")
            continue

        required = {COL_CELL_ID, COL_MISSION_ID, COL_MISSION_SEQUENCE_ORDER, COL_SOC_REFERENCE, COL_SOC_PREDICTION}
        missing = required - set(raw.columns)
        if missing:
            warnings.warn(f"[Vidal Fine-tuning] {path} missing {missing} -- skipping {cell_id}.")
            continue

        adapted = pd.DataFrame(
            {
                "framework": "Vidal Fine-tuning",
                COL_CELL_ID: raw[COL_CELL_ID],
                COL_MISSION_ID: raw[COL_MISSION_ID],
                COL_MISSION_SEQUENCE_ORDER: raw[COL_MISSION_SEQUENCE_ORDER].astype(int),
                COL_SOC_REFERENCE: raw[COL_SOC_REFERENCE],
                COL_SOC_PREDICTION: raw[COL_SOC_PREDICTION],
            }
        )
        if "elapsed_candidate_s_end" in raw.columns:
            adapted["elapsed_candidate_s"] = raw["elapsed_candidate_s_end"]
        if "original_time_s_end" in raw.columns:
            adapted["original_time_s"] = raw["original_time_s_end"]
        if COL_SOURCE_ROW_RIGHT in raw.columns:
            adapted[COL_SOURCE_ROW_RIGHT] = raw[COL_SOURCE_ROW_RIGHT]

        # Fallback-only local sample index per mission, ordered by elapsed
        # time when available -- never a fabricated "real" timestep_index.
        time_col = "elapsed_candidate_s" if "elapsed_candidate_s" in adapted.columns else None
        local_index = np.zeros(len(adapted), dtype=int)
        for _, group in adapted.groupby(COL_MISSION_ID, sort=False):
            order = (
                group[time_col].to_numpy().argsort()
                if time_col is not None
                else np.arange(len(group))
            )
            ranks = np.empty(len(order), dtype=int)
            ranks[order] = np.arange(len(order))
            local_index[group.index.to_numpy()] = ranks
        adapted[COL_TIMESTEP_INDEX] = local_index

        adapted[COL_SOC_REFERENCE] = _ensure_percent_scale(adapted[COL_SOC_REFERENCE])
        adapted[COL_SOC_PREDICTION] = _ensure_percent_scale(adapted[COL_SOC_PREDICTION])
        frames.append(adapted)

    if not frames:
        warnings.warn("[Vidal Fine-tuning] no cells produced usable data -- skipping.")
        return None
    return pd.concat(frames, ignore_index=True)


def load_all_predictions() -> dict[str, pd.DataFrame]:
    """One DataFrame per framework that has usable data, keyed by display
    name, returned in FRAMEWORK_ORDER. TATN is intentionally never
    included -- see the title cell."""
    result: dict[str, pd.DataFrame] = {}

    vidal = load_vidal_full_predictions()
    if vidal is not None:
        result["Vidal Fine-tuning"] = vidal

    dtnn = load_dtnn_predictions()
    if dtnn is not None:
        result["DTNN"] = dtnn

    for framework, path in GENERIC_RESULT_FILES.items():
        frame = _read_predictions_file(framework, path)
        if frame is not None:
            result[framework] = frame

    result.update(load_backbone_predictions())

    if not result:
        warnings.warn("No framework results were loadable -- check the config cell's paths.")
    return {framework: result[framework] for framework in FRAMEWORK_ORDER if framework in result}


all_predictions = load_all_predictions()

print("=== Loading summary ===")
print()
print("Vidal Fine-tuning:")
print(f"  root = {VIDAL_ROOT}")
print(f"  setting = {VIDAL_SETTING}")
print(f"  seed = {VIDAL_SEED} (baked into the path, no seed column)")
print(f"  cells loaded = {all_predictions['Vidal Fine-tuning'][COL_CELL_ID].nunique() if 'Vidal Fine-tuning' in all_predictions else 0} / {len(CELL_IDS)}")
print()
print("DTNN:")
print(f"  run_mode = {DTNN_RUN_MODE} (DTNN trajectory uses formal repeat 1)")
print(f"  method = {DTNN_METHOD}")
print(f"  repeat = {DTNN_REPEAT}")
print(f"  loaded = {'DTNN' in all_predictions}")
print()
print("SFTNN:")
print(f"  method = {FRAMEWORK_FILTER_SPECS['SFTNN'][COL_METHOD]}")
print(f"  seed = {PLOT_SEED}")
print(f"  loaded = {'SFTNN' in all_predictions}")
print()
print("DARM:")
print(f"  method = {FRAMEWORK_FILTER_SPECS['DARM'][COL_METHOD]}")
print(f"  seed = {PLOT_SEED}")
print(f"  loaded = {'DARM' in all_predictions}")
print()
print("Backbone:")
print(f"  models = {', '.join(BACKBONE_MODEL_TO_DISPLAY_NAME)}")
print(f"  method = {BACKBONE_METHOD}")
print(f"  budget = {BACKBONE_BUDGET}")
print(f"  seed = {PLOT_SEED}")
for _model_value, _display_name in BACKBONE_MODEL_TO_DISPLAY_NAME.items():
    print(f"  {_display_name}: loaded = {_display_name in all_predictions}")
print()
print("TATN: omitted -- no prediction-level output available (see title cell).")
print()
print("Loaded frameworks (plot order):", list(all_predictions.keys()))
for _framework, _frame in all_predictions.items():
    print(f"  {_framework}: {len(_frame):,} rows across {_frame[COL_CELL_ID].nunique()} cells")


In [ ]:
"""Code Cell 4 -- representative-mission selection.

Each subplot shows exactly one mission per cell, chosen deterministically:

1. **Anchor-based common-mission set.** With 7 curves of very different
   sparsity (Vidal/DARM are last_step_per_window; DTNN/SFTNN/Backbone are
   all_timesteps), requiring every loaded framework to share a mission
   would let the sparsest framework shrink or veto the representative
   mission for everyone. Instead, MISSION_ANCHOR_FRAMEWORKS
   (SFTNN, Backbone Transformer by default) determine the common mission
   set:
     - if >=1 anchor framework is loaded and has data for this cell, the
       common set is the intersection of ONLY the anchors' mission sets;
     - if none of the anchors are available for this cell, fall back to
       the intersection across every currently-loaded framework.
   Every OTHER framework is then plotted on the resulting mission if it
   has data there, and skipped (with a warning) if it does not -- a
   sparse framework never gets to change which mission was selected.
2. Sort the resulting common missions ascending by mission_sequence_order
   (numeric, never mission_cycle_id string order -- mission_cycle_id is
   only ever used as a display/diagnostic identifier).
3. MISSION_SELECTION_MODE == "midpoint" (the default): pick the mission at
   position (N - 1) // 2.
4. MISSION_OVERRIDE[cell_id], if set, takes priority over the midpoint
   rule; a forced mission that no framework has at all raises ValueError.

No `plot_x` / `mission_slot` / chronological concatenation / mission-
boundary logic remains -- a subplot draws one mission's real elapsed time,
nothing more.
"""


def _cell_mission_sets(cell_id: str, frameworks: list[str] | None = None) -> dict[str, set[int]]:
    result = {}
    source = frameworks if frameworks is not None else list(all_predictions)
    for framework in source:
        frame = all_predictions.get(framework)
        if frame is None:
            continue
        cell_frame = frame[frame[COL_CELL_ID] == cell_id]
        if cell_frame.empty:
            continue
        result[framework] = set(cell_frame[COL_MISSION_SEQUENCE_ORDER].dropna().astype(int).unique())
    return result


def _intersect(sets_by_framework: dict[str, set[int]]) -> set[int]:
    values = list(sets_by_framework.values())
    if not values:
        return set()
    common = values[0]
    for other in values[1:]:
        common = common & other
    return common


def _find_common_missions(cell_id: str) -> list[int]:
    """See the rule described in this cell's docstring: anchors first,
    full-intersection fallback only if no anchor is available."""
    anchor_sets = _cell_mission_sets(cell_id, MISSION_ANCHOR_FRAMEWORKS)
    if anchor_sets:
        return sorted(_intersect(anchor_sets))
    all_sets = _cell_mission_sets(cell_id)
    return sorted(_intersect(all_sets))


def select_representative_mission(cell_id: str) -> dict:
    """Returns {"mission_sequence_order", "mission_cycle_id", "position",
    "total"} for `cell_id`, printing the selection as required."""
    common_missions = _find_common_missions(cell_id)

    if cell_id in MISSION_OVERRIDE:
        forced = int(MISSION_OVERRIDE[cell_id])
        per_framework_sets = _cell_mission_sets(cell_id)
        present_anywhere = any(forced in missions for missions in per_framework_sets.values())
        if not present_anywhere:
            raise ValueError(
                f"MISSION_OVERRIDE[{cell_id!r}] = {forced} but no loaded framework has this "
                f"mission_sequence_order for {cell_id}."
            )
        missing_from = [fw for fw, missions in per_framework_sets.items() if forced not in missions]
        if missing_from:
            warnings.warn(
                f"MISSION_OVERRIDE[{cell_id!r}] = {forced} is missing from {missing_from}; "
                f"those frameworks will be omitted for this cell's subplot."
            )
        selected = forced
        total = len(common_missions) if common_missions else len(per_framework_sets)
        position = (common_missions.index(selected) + 1) if selected in common_missions else None
    else:
        if not common_missions:
            raise ValueError(f"No common test mission found across anchor frameworks for {cell_id}.")
        if MISSION_SELECTION_MODE != "midpoint":
            raise ValueError(f"Unsupported MISSION_SELECTION_MODE: {MISSION_SELECTION_MODE!r}")
        idx = (len(common_missions) - 1) // 2
        selected = common_missions[idx]
        total = len(common_missions)
        position = idx + 1

    mission_cycle_id = None
    for frame in all_predictions.values():
        match = frame[
            (frame[COL_CELL_ID] == cell_id) & (frame[COL_MISSION_SEQUENCE_ORDER] == selected)
        ]
        if not match.empty:
            mission_cycle_id = match[COL_MISSION_ID].iloc[0]
            break

    position_str = f"{position} / {total}" if position is not None and total else "n/a"
    print(
        f"{cell_id}:\n"
        f"  selected mission = {mission_cycle_id}\n"
        f"  mission_sequence_order = {selected}\n"
        f"  position = {position_str}"
    )
    return {
        "mission_sequence_order": selected,
        "mission_cycle_id": mission_cycle_id,
        "position": position,
        "total": total,
    }


def _resolve_time_column(frame: pd.DataFrame) -> str | None:
    for candidate in COL_ELAPSED_TIME_CANDIDATES:
        if candidate in frame.columns and frame[candidate].notna().any():
            return candidate
    return None


def _mission_frame(frame: pd.DataFrame, cell_id: str, mission_order: int) -> pd.DataFrame:
    return frame[
        (frame[COL_CELL_ID] == cell_id) & (frame[COL_MISSION_SEQUENCE_ORDER] == mission_order)
    ]


def _mission_xy(frame: pd.DataFrame, column: str, time_col: str | None) -> tuple[np.ndarray, np.ndarray]:
    """Real elapsed seconds on x (never min-max normalized), the raw
    per-framework prediction points on y -- no interpolation, no
    point-count matching across frameworks."""
    if time_col is not None:
        ordered = frame.sort_values(time_col)
        x = ordered[time_col].to_numpy(dtype=float)
    else:
        ordered = frame.sort_values(COL_TIMESTEP_INDEX)
        x = np.arange(len(ordered), dtype=float)
    y = ordered[column].to_numpy(dtype=float)
    return x, y


In [ ]:
"""Code Cell 5 -- plotting functions."""


def _downsample(x: np.ndarray, y: np.ndarray, every_n: int) -> tuple[np.ndarray, np.ndarray]:
    """Display-only thinning: keeps every `every_n`-th point and always
    forces in the last point. Never mutates the underlying results."""
    if every_n <= 1 or len(x) <= 1:
        return x, y
    idx = np.arange(0, len(x), every_n)
    if idx[-1] != len(x) - 1:
        idx = np.append(idx, len(x) - 1)
    return x[idx], y[idx]


def _choose_reference_framework(mission_frames: dict[str, pd.DataFrame]) -> str | None:
    """Prefer an all_timesteps framework (the most complete sampling of the
    mission) -- Vidal Fine-tuning and DARM (last_step_per_window) are
    deliberately never preferred, since they are far sparser. Among the
    eligible candidates (or, failing that, whatever is available), prefer
    the one with the most points for this mission."""
    if not mission_frames:
        return None
    all_timestep_candidates = [
        fw for fw in mission_frames if EVALUATION_UNIT.get(fw) == "all_timesteps"
    ]
    pool = all_timestep_candidates if all_timestep_candidates else list(mission_frames)
    return max(pool, key=lambda fw: len(mission_frames[fw]))


def check_reference_soc_consistency(
    mission_frames: dict[str, pd.DataFrame], reference_framework: str | None, cell_id: str
) -> None:
    """Cross-framework sanity check for the single selected mission.
    Matches observations by source_row_right when both frameworks have it
    (the exact, canonical join key -- Vidal Fine-tuning has this too, via
    the adapter); otherwise falls back to a nearest-time match on the
    shared elapsed-time column. Never requires equal row counts between
    frameworks (a schema difference, e.g. Vidal's last_step_per_window vs.
    SFTNN's all_timesteps, is not itself a mismatch) -- only the
    observations that can actually be matched are compared."""
    if reference_framework is None or len(mission_frames) < 2:
        return
    ref_frame = mission_frames[reference_framework]
    for framework, frame in mission_frames.items():
        if framework == reference_framework:
            continue
        if COL_SOURCE_ROW_RIGHT in ref_frame.columns and COL_SOURCE_ROW_RIGHT in frame.columns:
            merged = ref_frame[[COL_SOURCE_ROW_RIGHT, COL_SOC_REFERENCE]].merge(
                frame[[COL_SOURCE_ROW_RIGHT, COL_SOC_REFERENCE]],
                on=COL_SOURCE_ROW_RIGHT,
                how="inner",
                suffixes=("_ref", "_other"),
            )
        else:
            time_col = _resolve_time_column(ref_frame)
            if time_col is None or time_col not in frame.columns:
                continue
            merged = pd.merge_asof(
                ref_frame[[time_col, COL_SOC_REFERENCE]].sort_values(time_col),
                frame[[time_col, COL_SOC_REFERENCE]].sort_values(time_col),
                on=time_col,
                direction="nearest",
                suffixes=("_ref", "_other"),
                tolerance=1.0,
            ).dropna()
        if merged.empty:
            continue
        mismatch = ~np.isclose(
            merged[f"{COL_SOC_REFERENCE}_ref"],
            merged[f"{COL_SOC_REFERENCE}_other"],
            atol=REFERENCE_SOC_MISMATCH_TOLERANCE,
            equal_nan=True,
        )
        if mismatch.any():
            warnings.warn(
                f"[{cell_id}] reference SOC mismatch between {reference_framework} and "
                f"{framework} on {int(mismatch.sum())}/{len(mismatch)} matched observations."
            )


def plot_single_mission_soc(
    ax: plt.Axes,
    cell_id: str,
    mission_info: dict,
    *,
    every_n: int = PLOT_EVERY_N,
) -> tuple[dict[str, pd.DataFrame], str | None]:
    """Draws Reference SOC once, plus one line per framework that has data
    for this cell's selected mission, onto `ax`. Only line plots -- no
    fill_between / area fills. Does not create or show the figure."""
    mission_order = mission_info["mission_sequence_order"]

    mission_frames: dict[str, pd.DataFrame] = {}
    for framework in FRAMEWORK_ORDER:
        frame = all_predictions.get(framework)
        if frame is None:
            continue
        sub = _mission_frame(frame, cell_id, mission_order)
        if sub.empty:
            warnings.warn(
                f"[{cell_id}] {framework} has no rows for mission_sequence_order="
                f"{mission_order}; skipping this framework for this cell."
            )
            continue
        assert sub[COL_MISSION_SEQUENCE_ORDER].nunique() == 1, (
            f"[{cell_id}/{framework}] expected exactly one mission in this subplot's data"
        )
        mission_frames[framework] = sub

    reference_framework = _choose_reference_framework(mission_frames)
    check_reference_soc_consistency(mission_frames, reference_framework, cell_id)

    x_label = "Elapsed time within mission (s)"
    if reference_framework is not None:
        ref_frame = mission_frames[reference_framework]
        time_col_for_ref = _resolve_time_column(ref_frame)
        x_ref, y_ref = _mission_xy(ref_frame, COL_SOC_REFERENCE, time_col_for_ref)
        x_ref_d, y_ref_d = _downsample(x_ref, y_ref, every_n)
        ax.plot(
            x_ref_d, y_ref_d, color=REFERENCE_COLOR, linewidth=2.0, label="Reference SOC", zorder=5
        )

    for framework, sub in mission_frames.items():
        time_col = _resolve_time_column(sub)
        x, y = _mission_xy(sub, COL_SOC_PREDICTION, time_col)
        x_d, y_d = _downsample(x, y, every_n)
        ax.plot(
            x_d,
            y_d,
            color=FRAMEWORK_COLORS.get(framework),
            linewidth=1.4,
            alpha=0.9,
            label=framework,
        )
        if time_col is None:
            x_label = "Sample index within mission"

    ax.set_ylim(0, 100)
    ax.set_xlabel(x_label)
    ax.set_ylabel("SOC (%)")
    ax.set_title(cell_id)
    ax.annotate(
        f"Mission order: {mission_order}",
        xy=(0.98, 0.03),
        xycoords="axes fraction",
        ha="right",
        va="bottom",
        fontsize=7,
        color="#555555",
    )
    return mission_frames, reference_framework


def plot_all_cells_soc(mission_info_by_cell: dict[str, dict]) -> plt.Figure:
    """Combined 7-cell figure: a 3x6 GridSpec, first two rows split into
    three 2-column-wide subplots each, third row's single subplot centered
    in columns [2:4]. A shared, 2-row legend (8 entries: Reference SOC +
    7 frameworks) sits above all subplots so no per-subplot legend is
    needed."""
    fig = plt.figure(figsize=(16, 11))
    gs = gridspec.GridSpec(3, 6, figure=fig, hspace=0.45, wspace=0.3, top=0.86)
    slot_specs = [
        gs[0, 0:2], gs[0, 2:4], gs[0, 4:6],
        gs[1, 0:2], gs[1, 2:4], gs[1, 4:6],
        gs[2, 2:4],
    ]

    axes = []
    for cell_id, spec in zip(CELL_IDS, slot_specs):
        ax = fig.add_subplot(spec)
        plot_single_mission_soc(ax, cell_id, mission_info_by_cell[cell_id])
        axes.append(ax)

    handles_by_label: dict[str, object] = {}
    legend_order = ["Reference SOC", *FRAMEWORK_ORDER]
    for ax in axes:
        handles, labels = ax.get_legend_handles_labels()
        for handle, label in zip(handles, labels):
            handles_by_label.setdefault(label, handle)
    ordered_labels = [label for label in legend_order if label in handles_by_label]
    ordered_handles = [handles_by_label[label] for label in ordered_labels]

    fig.legend(
        ordered_handles,
        ordered_labels,
        loc="upper center",
        ncol=4,
        bbox_to_anchor=(0.5, 0.99),
        frameon=False,
    )
    fig.suptitle(
        "SOC Trajectories: Panasonic25 -> CMU eVTOL Transfer (one representative mission per cell)",
        y=1.03,
        fontsize=11,
    )
    return fig


In [ ]:
"""Diagnostics summary -- run before plotting.

Prints one row per cell: which mission was selected, how many prediction
points each loaded framework contributes for that mission. Point counts
are EXPECTED to differ across frameworks by design (Vidal/DARM are
last_step_per_window -- one point per window; DTNN/SFTNN/Backbone are
all_timesteps -- every raw timestep in every window); a low count is not a
bug."""

_diagnostic_rows = []
for _cell_id in CELL_IDS:
    _mission_info = select_representative_mission(_cell_id)
    _mission_frames = {
        framework: _mission_frame(frame, _cell_id, _mission_info["mission_sequence_order"])
        for framework, frame in all_predictions.items()
    }
    _mission_frames = {fw: sub for fw, sub in _mission_frames.items() if not sub.empty}
    _reference_framework = _choose_reference_framework(_mission_frames)

    for _sub in _mission_frames.values():
        assert _sub[COL_MISSION_SEQUENCE_ORDER].nunique() == 1

    _row = {
        "cell": _cell_id,
        "mission_order": _mission_info["mission_sequence_order"],
        "mission_cycle_id": _mission_info["mission_cycle_id"],
        "reference_source": _reference_framework,
    }
    for _framework in FRAMEWORK_ORDER:
        _row[f"{_framework}_points"] = len(_mission_frames.get(_framework, []))
    _diagnostic_rows.append(_row)

diagnostic_summary = pd.DataFrame(_diagnostic_rows)
print()
print(diagnostic_summary.to_string(index=False))


## Per-cell trajectories

Seven independent cells below (not a loop), one per target cell in
`CELL_IDS`, so each can be re-run/edited on its own. Each subplot shows
exactly one representative test mission (see `select_representative_mission`
above) -- not the whole test-set lifespan. TATN is omitted: no
prediction-level output is currently available (see the title cell).

In [ ]:
"""Code Cell 6 -- CELL_IDS[0]."""

_cell_id = CELL_IDS[0]
selected_mission = select_representative_mission(_cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
_mission_frames, _reference_framework = plot_single_mission_soc(ax, _cell_id, selected_mission)
assert len({sub[COL_MISSION_SEQUENCE_ORDER].iat[0] for sub in _mission_frames.values()}) <= 1
ax.legend(loc="upper right", frameon=False, fontsize=7)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


In [ ]:
"""Code Cell 7 -- CELL_IDS[1]."""

_cell_id = CELL_IDS[1]
selected_mission = select_representative_mission(_cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
_mission_frames, _reference_framework = plot_single_mission_soc(ax, _cell_id, selected_mission)
assert len({sub[COL_MISSION_SEQUENCE_ORDER].iat[0] for sub in _mission_frames.values()}) <= 1
ax.legend(loc="upper right", frameon=False, fontsize=7)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


In [ ]:
"""Code Cell 8 -- CELL_IDS[2]."""

_cell_id = CELL_IDS[2]
selected_mission = select_representative_mission(_cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
_mission_frames, _reference_framework = plot_single_mission_soc(ax, _cell_id, selected_mission)
assert len({sub[COL_MISSION_SEQUENCE_ORDER].iat[0] for sub in _mission_frames.values()}) <= 1
ax.legend(loc="upper right", frameon=False, fontsize=7)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


In [ ]:
"""Code Cell 9 -- CELL_IDS[3]."""

_cell_id = CELL_IDS[3]
selected_mission = select_representative_mission(_cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
_mission_frames, _reference_framework = plot_single_mission_soc(ax, _cell_id, selected_mission)
assert len({sub[COL_MISSION_SEQUENCE_ORDER].iat[0] for sub in _mission_frames.values()}) <= 1
ax.legend(loc="upper right", frameon=False, fontsize=7)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


In [ ]:
"""Code Cell 10 -- CELL_IDS[4]."""

_cell_id = CELL_IDS[4]
selected_mission = select_representative_mission(_cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
_mission_frames, _reference_framework = plot_single_mission_soc(ax, _cell_id, selected_mission)
assert len({sub[COL_MISSION_SEQUENCE_ORDER].iat[0] for sub in _mission_frames.values()}) <= 1
ax.legend(loc="upper right", frameon=False, fontsize=7)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


In [ ]:
"""Code Cell 11 -- CELL_IDS[5]."""

_cell_id = CELL_IDS[5]
selected_mission = select_representative_mission(_cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
_mission_frames, _reference_framework = plot_single_mission_soc(ax, _cell_id, selected_mission)
assert len({sub[COL_MISSION_SEQUENCE_ORDER].iat[0] for sub in _mission_frames.values()}) <= 1
ax.legend(loc="upper right", frameon=False, fontsize=7)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


In [ ]:
"""Code Cell 12 -- CELL_IDS[6]."""

_cell_id = CELL_IDS[6]
selected_mission = select_representative_mission(_cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
_mission_frames, _reference_framework = plot_single_mission_soc(ax, _cell_id, selected_mission)
assert len({sub[COL_MISSION_SEQUENCE_ORDER].iat[0] for sub in _mission_frames.values()}) <= 1
ax.legend(loc="upper right", frameon=False, fontsize=7)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


## Combined figure (all seven cells)

In [ ]:
"""Code Cell 13 -- combined 3+3+1 figure across all seven cells."""

_mission_info_by_cell = {cell_id: select_representative_mission(cell_id) for cell_id in CELL_IDS}
combined_fig = plot_all_cells_soc(_mission_info_by_cell)
plt.show()


## Export

In [ ]:
"""Code Cell 14 -- export the combined figure to PNG and PDF."""

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
png_path = OUTPUT_DIR / "soc_trajectory_all_cells.png"
pdf_path = OUTPUT_DIR / "soc_trajectory_all_cells.pdf"
combined_fig.savefig(png_path, dpi=FIG_DPI, bbox_inches="tight")
combined_fig.savefig(pdf_path, dpi=FIG_DPI, bbox_inches="tight")
print(f"Saved: {png_path}")
print(f"Saved: {pdf_path}")
